# Étude de cas : ventes et livraisons

Une entreprise vend des produits en ligne (épicerie, frais, boissons, hygiène, maison, électronique, livres, bricolage, collection) et les livre elle-même, depuis ses entrepôts de la région parisienne, avec sa propre flotte de véhicules.

### Les fichiers (`/work/data`)

| Fichier | Contenu |
|---|---|
| `produits.csv` | le catalogue : `produit_id, nom, categorie, sous_categorie, prix, poids_kg` |
| `commandes.csv` | le journal des commandes, une ligne par événement : `commande_id, client_id, horodatage, evenement, livreur_id, detail` |
| `vehicules/<vehicule>/<AAAA-MM>.csv` | la trace GPS de chaque véhicule, **un point par minute** : `timestamp, livreur_id, lon, lat, vitesse_kmh, dans_vehicule, charge_kg` |
| `communes.csv` | les communes de Paris et de la petite couronne et les 20 arrondissements de Paris, avec leur centre : `commune, code_postal, lon, lat` |
| `communes.geojson` | les contours des mêmes communes (propriétés `commune`, `code_postal`) |

### Le journal des commandes

Une commande apparaît plusieurs fois : une ligne par étape.

| `evenement` | Sens | `livreur_id` | `detail` |
|---|---|---|---|
| `commande` | le client passe commande | — | les produits : `P0201x1\|P0033x4` (produit `x` quantité) |
| `chargee` | la commande est chargée dans un véhicule | le livreur | — |
| `livree` | le client signe ; horodatage à la seconde | le livreur | — |
| `echec` | la livraison n'a pas pu se faire | le livreur | le motif |
| `reportee` | chargée mais pas tentée : rapportée à l'entrepôt en fin de service | le livreur | — |
| `retour` | la commande est abandonnée et retournée à l'entrepôt | le livreur | — |

### Les traces GPS

Un livreur conduit un seul véhicule par jour. `dans_vehicule` vaut `false` quand le livreur est descendu du véhicule ; `charge_kg` est le poids transporté.

### Les services

| Adresse | Service |
|---|---|
| http://localhost:8888 | JupyterLab |
| http://localhost:8080 | Master Spark |
| http://localhost:4040 | Spark UI de la session ouverte |
| http://localhost:18080 | Spark History |
| `mongodb://mongo:27017` | MongoDB |

## 1) Charger la donnée

1.1) La session Spark et les fonctions fournies.

In [45]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("livraisons")
         .master("spark://spark-master:7077")
         .getOrCreate())
DATA = "/work/data"
RESULTATS = "/work/resultats"
spark.sparkContext.setLogLevel("ERROR")


def haversine(lon1, lat1, lon2, lat2):
    """Distance en km entre deux points : noms de colonnes ou colonnes, en degrés."""
    lon1, lat1, lon2, lat2 = [F.col(c) if isinstance(c, str) else c for c in (lon1, lat1, lon2, lat2)]
    dlon = F.radians(lon2 - lon1)
    dlat = F.radians(lat2 - lat1)
    a = F.sin(dlat / 2) ** 2 + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlon / 2) ** 2
    return 2 * 6371.0 * F.asin(F.sqrt(a))

1.2) Définissez un schéma explicite (`StructType`) pour chacun des 4 fichiers CSV, puis chargez `produits.csv`, `commandes.csv`, `communes.csv` et **tous** les fichiers GPS en une seule lecture (`vehicules/*/*.csv`). Les horodatages doivent être de type `timestamp`, `dans_vehicule` de type `boolean`, les nombres de type `double`.

Affichez le nombre de lignes de chaque DataFrame.



In [31]:
schema_produits = T.StructType([
    T.StructField("produit_id", T.StringType(), True),
    T.StructField("nom", T.StringType(), True),
    T.StructField("categorie", T.StringType(), True),
    T.StructField("sous_categorie", T.StringType(), True),
    T.StructField("prix", T.DoubleType(), True),
    T.StructField("poids_kg", T.DoubleType(), True)

])

schema_communes = T.StructType([
    T.StructField("commune", T.StringType(), True),
    T.StructField("code_postal", T.StringType(), True),
    T.StructField("lon", T.DoubleType(), True),
    T.StructField("lat", T.DoubleType(), True)
])

schema_commandes = T.StructType([
    T.StructField("commande_id", T.StringType(), True),
    T.StructField("client_id", T.StringType(), True),
    T.StructField("horodatage", T.TimestampType(), True),
    T.StructField("evenement", T.StringType(), True),
    T.StructField("livreur_id", T.StringType(), True),
    T.StructField("detail", T.StringType(), True)

])

schema_gps = T.StructType([
    T.StructField("timestamp", T.TimestampType(), True),
    T.StructField("livreur_id", T.StringType(), True),
    T.StructField("lon", T.DoubleType(), True),
    T.StructField("lat", T.DoubleType(), True),
    T.StructField("vitesse_kmh", T.DoubleType(), True),
    T.StructField("dans_vehicule", T.BooleanType(), True),
    T.StructField("charge_kg", T.DoubleType(), True)
])

produits = (
    spark.read
    .option("header", True)
    .schema(schema_produits)
    .csv(f"{DATA}/produits.csv")
)

communes = (
    spark.read
    .option("header", True)
    .schema(schema_communes)
    .csv(f"{DATA}/communes.csv")
)

commandes = (
    spark.read
    .option("header", True)
    .schema(schema_commandes)
    .csv(f"{DATA}/commandes.csv")
)

gps = (
    spark.read
    .option("header", True)
    .schema(schema_gps)
    .csv(f"{DATA}/vehicules/*/*.csv")
)

for nom, df in [
    ("produits", produits),
    ("communes", communes),
    ("commandes", commandes),
    ("gps", gps),
]:
    print(f"{nom} : {df.count()} lignes")

produits : 233 lignes
communes : 142 lignes
commandes : 846535 lignes


gps : 5174905 lignes


1.3) L'identifiant du véhicule n'est pas une colonne des fichiers GPS : il est dans le chemin du fichier. Ajoutez une colonne `vehicule_id` (fonction `F.input_file_name()`), et une colonne `jour` (la date du point).

In [32]:
gps = (
    gps
    .withColumn(
        "vehicule_id",
        F.regexp_extract(
            F.input_file_name(),
            r"/vehicules/([^/]+)/",
            1
        )
    )
    .withColumn("jour", F.to_date("timestamp"))
)

gps.select("vehicule_id").distinct().orderBy("vehicule_id").show(10)

gps.select("vehicule_id", "timestamp", "jour").distinct().orderBy("vehicule_id").show(10)

+-----------+
|vehicule_id|
+-----------+
|        V01|
|        V02|
|        V03|
|        V04|
|        V05|
|        V06|
|        V07|
|        V08|
|        V09|
|        V10|
+-----------+
only showing top 10 rows



+-----------+-------------------+----------+
|vehicule_id|          timestamp|      jour|
+-----------+-------------------+----------+
|        V01|2025-01-02 15:30:00|2025-01-02|
|        V01|2025-01-02 08:10:00|2025-01-02|
|        V01|2025-01-06 12:54:00|2025-01-06|
|        V01|2025-01-04 13:27:00|2025-01-04|
|        V01|2025-01-03 07:59:00|2025-01-03|
|        V01|2025-01-02 13:32:00|2025-01-02|
|        V01|2025-01-03 08:27:00|2025-01-03|
|        V01|2025-01-02 15:14:00|2025-01-02|
|        V01|2025-01-03 09:19:00|2025-01-03|
|        V01|2025-01-03 08:35:00|2025-01-03|
+-----------+-------------------+----------+
only showing top 10 rows



## 2) Vérifier la donnée

2.1) Combien y a-t-il de commandes, de clients, de livreurs et de véhicules ?

In [33]:
nb_commandes = commandes.select("commande_id").distinct().count()

nb_clients = commandes.select("client_id").distinct().count()

nb_livreurs = (
    commandes
    .filter(F.col("livreur_id").isNotNull())
    .select("livreur_id")
    .distinct()
    .count()
)

nb_vehicules = gps.select("vehicule_id").distinct().count()

print(f"Nombre de commandes : {nb_commandes}")
print(f"Nombre de clients : {nb_clients}")
print(f"Nombre de livreurs : {nb_livreurs}")
print(f"Nombre de véhicules : {nb_vehicules}")

Nombre de commandes : 271617
Nombre de clients : 8000
Nombre de livreurs : 90
Nombre de véhicules : 72


2.2) Comptez les événements du journal par type.

In [34]:
commandes.groupBy("evenement").count().orderBy("evenement").show()

+---------+------+
|evenement| count|
+---------+------+
|  chargee|286862|
| commande|271617|
|    echec| 11220|
|   livree|268682|
| reportee|  6960|
|   retour|  1194|
+---------+------+



2.3) Pour chaque commande, comptez ses événements `chargee`. Combien de commandes ont été chargées 0, 1, 2, 3… fois ?

In [35]:
chargements_par_commande = (
    commandes
    .groupBy("commande_id")
    .agg(
        F.count(
            F.when(F.col("evenement") == "chargee", 1)
        ).alias("nb_chargements")
    )
)

(
    chargements_par_commande
    .groupBy("nb_chargements")
    .agg(F.count("*").alias("nb_commandes"))
    .orderBy("nb_chargements")
    .show()
)

+--------------+------------+
|nb_chargements|nb_commandes|
+--------------+------------+
|             0|        1637|
|             1|      254229|
|             2|       14673|
|             3|        1026|
|             4|          51|
|             5|           1|
+--------------+------------+



2.4) Pour chaque événement `chargee`, trouvez l'événement qui le précède dans la vie de la même commande. Comptez les `chargee` par événement précédent.

In [36]:
fenetre = Window.partitionBy("commande_id").orderBy("horodatage")

commandes_precedent = commandes.withColumn(
    "evenement_precedent",
    F.lag("evenement", 1).over(fenetre)
)

(
    commandes_precedent
    .filter(F.col("evenement") == "chargee")
    .groupBy("evenement_precedent")
    .count()
    .withColumnRenamed("count", "nb_chargements")
    .orderBy("evenement_precedent")
    .show()
)

+-------------------+--------------+
|evenement_precedent|nb_chargements|
+-------------------+--------------+
|           commande|        269980|
|              echec|          9950|
|           reportee|          6932|
+-------------------+--------------+



*Un rechargement suit toujours un `echec` (nouvelle tentative) ou un `reportee` (commande rapportée en fin de service) : il y a plus de `chargee` que de `commande`.*

2.5) Au 31/12/2025, combien de commandes ne sont pas livrées ? Séparez les commandes abandonnées (événement `retour`) des commandes encore en cours.

In [37]:
commandes_2025 = commandes.filter(
    F.col("horodatage") < F.lit("2026-01-01").cast("timestamp")
)

fenetre = (
    Window
    .partitionBy("commande_id")
    .orderBy(F.col("horodatage").desc())
)

dernier_evenement = (
    commandes_2025
    .withColumn("rang", F.row_number().over(fenetre))
    .filter(F.col("rang") == 1)
)

commandes_non_livrees = (
    dernier_evenement
    .filter(F.col("evenement") != "livree")
    .withColumn(
        "statut",
        F.when(F.col("evenement") == "retour", "abandonnee")
        .otherwise("en cours")
    )
)

print(f"Commandes non livrées : {commandes_non_livrees.count()}")

commandes_non_livrees.groupBy("statut").count().show()

Commandes non livrées : 2935


+----------+-----+
|    statut|count|
+----------+-----+
|  en cours| 1741|
|abandonnee| 1194|
+----------+-----+



## 3) Les lignes de commande

3.1) Les produits d'une commande sont dans la colonne `detail` de l'événement `commande`. Construisez la table `lignes` : une ligne par produit commandé, avec `commande_id`, `client_id`, `horodatage`, `produit_id`, `quantite`.

In [38]:
lignes = commandes.filter(F.col("evenement") == "commande")

lignes = lignes.withColumn(
    "liste_produits",
    F.split(F.col("detail"), r"\|")
)

lignes = lignes.withColumn(
    "produit_quantite",
    F.explode(F.col("liste_produits"))
)

lignes = lignes.withColumn(
    "produit_id",
    F.split(F.col("produit_quantite"), "x").getItem(0)
)

lignes = lignes.withColumn(
    "quantite",
    F.split(F.col("produit_quantite"), "x").getItem(1).cast("int")
)

lignes = lignes.select(
    "commande_id",
    "client_id",
    "horodatage",
    "produit_id",
    "quantite"
)

lignes.show(10)

+-----------+---------+-------------------+----------+--------+
|commande_id|client_id|         horodatage|produit_id|quantite|
+-----------+---------+-------------------+----------+--------+
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0037|       5|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0063|       1|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0019|       2|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0067|       1|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0069|       2|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0089|       3|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0061|       1|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0055|       2|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0009|       3|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0076|       1|
+-----------+---------+-------------------+----------+--------+
only showing top 10 rows



3.2) Ajoutez à `lignes` les colonnes du catalogue et le `montant` de chaque ligne. Construisez la table `commandes` : une ligne par commande, avec `montant`, `poids_kg` et `nb_lignes`.

In [39]:
lignes = lignes.join(
    produits,
    on="produit_id",
    how="left"
)

lignes = lignes.withColumn(
    "montant",
    F.col("quantite") * F.col("prix")
)

commandes_resume = (
    lignes
    .groupBy("commande_id", "client_id", "horodatage")
    .agg(
        F.sum("montant").alias("montant"),
        F.sum(
            F.col("quantite") * F.col("poids_kg")
        ).alias("poids_kg"),
        F.count("*").alias("nb_lignes")
    )
)

lignes.show(10)
commandes_resume.show(10)

+----------+-----------+---------+-------------------+--------+--------------------+---------+-------------------+----+--------+------------------+
|produit_id|commande_id|client_id|         horodatage|quantite|                 nom|categorie|     sous_categorie|prix|poids_kg|           montant|
+----------+-----------+---------+-------------------+--------+--------------------+---------+-------------------+----+--------+------------------+
|     P0037|   C0000001|  CL05658|2025-01-01 00:03:44|       5|Cookies chocolat ...| Épicerie| Biscuits et snacks| 2.6|     0.3|              13.0|
|     P0063|   C0000001|  CL05658|2025-01-01 00:03:44|       1| Lardons fumés 200 g|    Frais|Viandes et poissons| 2.4|     0.2|               2.4|
|     P0019|   C0000001|  CL05658|2025-01-01 00:03:44|       2|   Capsules café x30| Épicerie|     Petit-déjeuner| 8.5|     0.2|              17.0|
|     P0067|   C0000001|  CL05658|2025-01-01 00:03:44|       1|        Bananes 1 kg|    Frais|  Fruits et légume

+-----------+---------+-------------------+------------------+------------------+---------+
|commande_id|client_id|         horodatage|           montant|          poids_kg|nb_lignes|
+-----------+---------+-------------------+------------------+------------------+---------+
|   C0000168|  CL06333|2025-01-01 13:16:07|            209.35| 60.97999999999999|       32|
|   C0000469|  CL03079|2025-01-01 22:39:48| 80.89999999999998|             14.03|       13|
|   C0000752|  CL00230|2025-01-02 16:34:53|            218.05|            46.515|       31|
|   C0000810|  CL07244|2025-01-02 18:29:22|239.45000000000002| 78.49999999999999|       27|
|   C0000855|  CL04907|2025-01-02 19:48:36| 70.89999999999999|             15.87|       15|
|   C0000874|  CL05590|2025-01-02 20:11:48| 75.49999999999999|             14.47|       15|
|   C0001297|  CL04178|2025-01-03 18:18:41| 69.89999999999999|             28.41|       15|
|   C0001491|  CL05514|2025-01-03 22:54:31|242.55000000000007|49.529999999999994

## 4) Les entrepôts

4.1) Pour chaque véhicule et chaque jour, gardez le premier et le dernier point GPS. Arrondissez leurs coordonnées à 0,01° et comptez les journées par position arrondie. Combien d'entrepôts y a-t-il ?

In [43]:
fenetre_debut = (
    Window
    .partitionBy("vehicule_id", "jour")
    .orderBy("timestamp")
)

fenetre_fin = (
    Window
    .partitionBy("vehicule_id", "jour")
    .orderBy(F.col("timestamp").desc())
)

gps_classes = (
    gps
    .withColumn("rang_debut", F.row_number().over(fenetre_debut))
    .withColumn("rang_fin", F.row_number().over(fenetre_fin))
)

positions_journalieres = (
    gps_classes
    .filter(
        (F.col("rang_debut") == 1) | (F.col("rang_fin") == 1)
    )
    .select(
        "vehicule_id",
        "jour",
        F.round("lon", 2).alias("lon_arrondie"),
        F.round("lat", 2).alias("lat_arrondie")
    )
    .distinct()
)

positions_entrepots = (
    positions_journalieres
    .groupBy("lon_arrondie", "lat_arrondie")
    .count()
    .withColumnRenamed("count", "nb_journees")
    .orderBy(F.col("nb_journees").desc())
)

print(f"Nombre d'entrepôts : {positions_entrepots.count()}")

Nombre d'entrepôts : 3


4.2) Construisez la table `entrepots` (`entrepot`, `lon`, `lat`) : `E1`, `E2`… par longitude croissante, position = moyenne des premiers points. Construisez la table `vehicule_entrepot` (`vehicule_id`, `entrepot`) et comptez les véhicules par entrepôt.

In [ ]:
premiers_points = (
    gps_classes
    .filter(F.col("rang_debut") == 1)
    .withColumn("lon_arrondie", F.round("lon", 2))
    .withColumn("lat_arrondie", F.round("lat", 2))
)

positions_moyennes = (
    premiers_points
    .groupBy("lon_arrondie", "lat_arrondie")
    .agg(
        F.avg("lon").alias("lon"),
        F.avg("lat").alias("lat")
    )
)

ordre_entrepots = Window.orderBy("lon", "lat")

correspondance_entrepots = (
    positions_moyennes
    .withColumn("numero", F.row_number().over(ordre_entrepots))
    .withColumn(
        "entrepot",
        F.concat(F.lit("E"), F.col("numero").cast("string"))
    )
)

entrepots = correspondance_entrepots.select(
    "entrepot", "lon", "lat"
)

vehicule_entrepot = (
    premiers_points
    .select("vehicule_id", "lon_arrondie", "lat_arrondie")
    .distinct()
    .join(
        correspondance_entrepots.select(
            "lon_arrondie", "lat_arrondie", "entrepot"
        ),
        on=["lon_arrondie", "lat_arrondie"],
        how="inner"
    )
    .select("vehicule_id", "entrepot")
    .distinct()
)

entrepots.orderBy("lon").show()

(
    vehicule_entrepot
    .groupBy("entrepot")
    .count()
    .withColumnRenamed("count", "nb_vehicules")
    .orderBy("entrepot")
    .show()
)

+--------+------------------+------------------+
|entrepot|               lon|               lat|
+--------+------------------+------------------+
|      E1| 2.277068913341649|48.936712904613536|
|      E2| 2.351513594202888|48.747046985844335|
|      E3|2.4489591059995286|48.906701645482606|
+--------+------------------+------------------+



+--------+------------+
|entrepot|nb_vehicules|
+--------+------------+
|      E1|          24|
|      E2|          24|
|      E3|          24|
+--------+------------+



## 5) Les livraisons

On découpe la journée de chaque livreur en **livraisons** : une livraison commence quand le livreur descend du véhicule et dure jusqu'à la livraison suivante. Le chargement du matin, les retours à l'entrepôt et la pause déjeuner sont aussi des périodes à pied : la question 5.6 les distingue.

5.1) Ajoutez à `gps` une colonne `changement` : 1 si le livreur vient de descendre du véhicule (point précédent du même livreur, le même jour, `dans_vehicule = true` ; point courant `false`), 0 sinon. Le premier point de la journée vaut 0.

Affichez les lignes où `changement = 1` pour le livreur `L001` le 2025-01-06.

In [47]:
fenetre = (
    Window
    .partitionBy("livreur_id", "jour")
    .orderBy("timestamp")
)

gps = gps.withColumn(
    "dans_vehicule_precedent",
    F.lag("dans_vehicule", 1).over(fenetre)
)

gps = gps.withColumn(
    "changement",
    F.when(
        (F.col("dans_vehicule_precedent") == True)
        & (F.col("dans_vehicule") == False),
        1
    ).otherwise(0)
)

(
    gps
    .filter(
        (F.col("livreur_id") == "L001")
        & (F.col("jour") == F.lit("2025-01-06").cast("date"))
        & (F.col("changement") == 1)
    )
    .select(
        "livreur_id",
        "jour",
        "timestamp",
        "dans_vehicule_precedent",
        "dans_vehicule",
        "changement"
    )
    .orderBy("timestamp")
    .show(100, truncate=False)
)

+----------+----------+-------------------+-----------------------+-------------+----------+
|livreur_id|jour      |timestamp          |dans_vehicule_precedent|dans_vehicule|changement|
+----------+----------+-------------------+-----------------------+-------------+----------+
|L001      |2025-01-06|2025-01-06 09:54:00|true                   |false        |1         |
|L001      |2025-01-06|2025-01-06 10:00:00|true                   |false        |1         |
|L001      |2025-01-06|2025-01-06 10:09:00|true                   |false        |1         |
|L001      |2025-01-06|2025-01-06 10:23:00|true                   |false        |1         |
|L001      |2025-01-06|2025-01-06 10:36:00|true                   |false        |1         |
|L001      |2025-01-06|2025-01-06 10:54:00|true                   |false        |1         |
|L001      |2025-01-06|2025-01-06 11:12:00|true                   |false        |1         |
|L001      |2025-01-06|2025-01-06 11:38:00|true                   |fal

5.2) Ajoutez une colonne `numero_livraison` : la somme cumulée de `changement` pour chaque livreur et chaque jour. Elle vaut 0 au chargement du matin.

Pour `L001` le 2025-01-06, affichez par `numero_livraison` le premier et le dernier timestamp, et le nombre de minutes à pied.

In [48]:
fenetre_cumulative = (
    Window
    .partitionBy("livreur_id", "jour")
    .orderBy("timestamp")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

gps = gps.withColumn(
    "numero_livraison",
    F.sum("changement").over(fenetre_cumulative)
)

(
    gps
    .filter(
        (F.col("livreur_id") == "L001")
        & (F.col("jour") == F.lit("2025-01-06").cast("date"))
    )
    .groupBy("numero_livraison")
    .agg(
        F.min("timestamp").alias("premier_timestamp"),
        F.max("timestamp").alias("dernier_timestamp"),
        F.count(
            F.when(F.col("dans_vehicule") == False, 1)
        ).alias("minutes_a_pied")
    )
    .orderBy("numero_livraison")
    .show(100, truncate=False)
)

+----------------+-------------------+-------------------+--------------+
|numero_livraison|premier_timestamp  |dernier_timestamp  |minutes_a_pied|
+----------------+-------------------+-------------------+--------------+
|0               |2025-01-06 07:43:00|2025-01-06 09:53:00|44            |
|1               |2025-01-06 09:54:00|2025-01-06 09:59:00|4             |
|2               |2025-01-06 10:00:00|2025-01-06 10:08:00|7             |
|3               |2025-01-06 10:09:00|2025-01-06 10:22:00|10            |
|4               |2025-01-06 10:23:00|2025-01-06 10:35:00|4             |
|5               |2025-01-06 10:36:00|2025-01-06 10:53:00|8             |
|6               |2025-01-06 10:54:00|2025-01-06 11:11:00|4             |
|7               |2025-01-06 11:12:00|2025-01-06 11:37:00|18            |
|8               |2025-01-06 11:38:00|2025-01-06 12:01:00|17            |
|9               |2025-01-06 12:02:00|2025-01-06 12:14:00|4             |
|10              |2025-01-06 12:15:00|

5.3) Construisez la table `livraisons` : une ligne par (`livreur_id`, `jour`, `numero_livraison`), calculée sur les seuls points où le livreur est à pied, avec `vehicule_id`, `debut`, `fin`, `duree_min` (nombre de points) et la position moyenne `lon`, `lat`. Combien de lignes contient-elle ?

In [49]:
livraisons = (
    gps
    .filter(F.col("dans_vehicule") == False)
    .groupBy("livreur_id", "jour", "numero_livraison")
    .agg(
        F.first("vehicule_id").alias("vehicule_id"),
        F.min("timestamp").alias("debut"),
        F.max("timestamp").alias("fin"),
        F.count("*").alias("duree_min"),
        F.avg("lon").alias("lon"),
        F.avg("lat").alias("lat")
    )
)

print(f"Nombre de livraisons : {livraisons.count()}")

livraisons.orderBy("livreur_id", "jour", "numero_livraison").show(
    10, truncate=False
)

Nombre de livraisons : 273593


+----------+----------+----------------+-----------+-------------------+-------------------+---------+------------------+------------------+
|livreur_id|jour      |numero_livraison|vehicule_id|debut              |fin                |duree_min|lon               |lat               |
+----------+----------+----------------+-----------+-------------------+-------------------+---------+------------------+------------------+
|L001      |2025-01-06|0               |V10        |2025-01-06 07:43:00|2025-01-06 08:26:00|44       |2.277131022727271 |48.93670315909091 |
|L001      |2025-01-06|1               |V10        |2025-01-06 09:54:00|2025-01-06 09:57:00|4        |2.311953          |48.87138975       |
|L001      |2025-01-06|2               |V10        |2025-01-06 10:00:00|2025-01-06 10:06:00|7        |2.313446714285714 |48.87084757142858 |
|L001      |2025-01-06|3               |V10        |2025-01-06 10:09:00|2025-01-06 10:18:00|10       |2.3160128999999996|48.87292010000001 |
|L001      |2

5.4) Ajoutez à chaque livraison ses signatures et ses échecs. Un événement `livree` ou `echec` appartient à une livraison quand il a le même livreur, le même jour, et un horodatage compris entre `debut` et `fin`, avec 1 minute de marge de chaque côté : les points GPS sont à la minute, les événements à la seconde.

Construisez la table `livraisons_ev` : les colonnes de `livraisons`, plus `signature` (horodatage de l'événement), `evenement`, `commande_id` et `client_id`. Une ligne par événement ; une livraison sans événement garde une ligne, avec ces 4 colonnes vides.

In [50]:
evenements_livraison = (
    commandes
    .filter(F.col("evenement").isin("livree", "echec"))
    .select(
        "livreur_id",
        F.to_date("horodatage").alias("jour"),
        F.col("horodatage").alias("signature"),
        "evenement",
        "commande_id",
        "client_id"
    )
)

livraisons_ev = (
    livraisons.alias("l")
    .join(
        evenements_livraison.alias("e"),
        (
            (F.col("l.livreur_id") == F.col("e.livreur_id"))
            & (F.col("l.jour") == F.col("e.jour"))
            & (
                F.col("e.signature")
                >= F.col("l.debut") - F.expr("INTERVAL 1 MINUTE")
            )
            & (
                F.col("e.signature")
                <= F.col("l.fin") + F.expr("INTERVAL 1 MINUTE")
            )
        ),
        how="left"
    )
    .select(
        "l.*",
        "e.signature",
        "e.evenement",
        "e.commande_id",
        "e.client_id"
    )
)

livraisons_ev.orderBy(
    "livreur_id", "jour", "numero_livraison", "signature"
).show(20, truncate=False)

+----------+----------+----------------+-----------+-------------------+-------------------+---------+------------------+------------------+-------------------+---------+-----------+---------+
|livreur_id|jour      |numero_livraison|vehicule_id|debut              |fin                |duree_min|lon               |lat               |signature          |evenement|commande_id|client_id|
+----------+----------+----------------+-----------+-------------------+-------------------+---------+------------------+------------------+-------------------+---------+-----------+---------+
|L001      |2025-01-06|0               |V10        |2025-01-06 07:43:00|2025-01-06 08:26:00|44       |2.277131022727271 |48.93670315909091 |NULL               |NULL     |NULL       |NULL     |
|L001      |2025-01-06|1               |V10        |2025-01-06 09:54:00|2025-01-06 09:57:00|4        |2.311953          |48.87138975       |2025-01-06 09:54:50|livree   |C0001606   |CL07994  |
|L001      |2025-01-06|2           

5.5) Combien d'événements `livree` et `echec` du journal ne sont rattachés à aucune livraison ?

In [ ]:
evenements_rattaches = (
    livraisons_ev
    .filter(F.col("evenement").isNotNull())
    .select("livreur_id", "commande_id", "evenement", "signature")
    .distinct()
)

evenements_non_rattaches = (
    evenements_livraison
    .join(
        evenements_rattaches,
        on=["livreur_id", "commande_id", "evenement", "signature"],
        how="left_anti"
    )
)

print(f"Total non rattaché : {evenements_non_rattaches.count()}")

+---------+-----+
|evenement|count|
+---------+-----+
+---------+-----+



Total non rattaché : 0


5.6) Toutes les périodes à pied ne sont pas des livraisons. Ajoutez à `livraisons_ev` une colonne `type` :
- `livraison` : au moins un événement ;
- `entrepot` : sinon, position à moins de 300 m d'un entrepôt ;
- `pause` : les autres.

Pour chaque type : nombre de lignes de `livraisons`, durée moyenne, heure moyenne de début.

In [52]:
distances_entrepots = (
    livraisons.alias("l")
    .crossJoin(entrepots.alias("e"))
    .select(
        "l.livreur_id",
        "l.jour",
        "l.numero_livraison",
        haversine("l.lon", "l.lat", "e.lon", "e.lat").alias("distance_km")
    )
    .groupBy("livreur_id", "jour", "numero_livraison")
    .agg(F.min("distance_km").alias("distance_entrepot_km"))
)

livraisons_ev = (
    livraisons_ev
    .join(
        distances_entrepots,
        on=["livreur_id", "jour", "numero_livraison"],
        how="left"
    )
    .withColumn(
        "type",
        F.when(F.col("evenement").isNotNull(), "livraison")
        .when(F.col("distance_entrepot_km") < 0.3, "entrepot")
        .otherwise("pause")
    )
    .drop("distance_entrepot_km")
)

livraisons_par_type = (
    livraisons_ev
    .select(
        "livreur_id", "jour", "numero_livraison",
        "type", "debut", "duree_min"
    )
    .dropDuplicates(["livreur_id", "jour", "numero_livraison"])
    .withColumn(
        "heure_debut",
        F.hour("debut")
        + F.minute("debut") / 60
        + F.second("debut") / 3600
    )
)

(
    livraisons_par_type
    .groupBy("type")
    .agg(
        F.count("*").alias("nb_livraisons"),
        F.round(F.avg("duree_min"), 2).alias("duree_moyenne_min"),
        F.round(F.avg("heure_debut"), 2).alias("heure_moyenne_debut")
    )
    .orderBy("type")
    .show(truncate=False)
)

+---------+-------------+-----------------+-------------------+
|type     |nb_livraisons|duree_moyenne_min|heure_moyenne_debut|
+---------+-------------+-----------------+-------------------+
|entrepot |22368        |27.21            |11.68              |
|livraison|246258       |8.32             |12.19              |
|pause    |4967         |37.5             |12.35              |
+---------+-------------+-----------------+-------------------+



5.7) Une même livraison peut compter plusieurs événements : plusieurs commandes à la même adresse, remises à pied l'une après l'autre. Combien de livraisons ont 1, 2, 3… événements ? Combien concernent au moins 2 clients différents ?

In [53]:
evenements_par_livraison = (
    livraisons_ev
    .filter(F.col("evenement").isNotNull())
    .groupBy("livreur_id", "jour", "numero_livraison")
    .agg(
        F.count("*").alias("nb_evenements"),
        F.countDistinct("client_id").alias("nb_clients")
    )
)

(
    evenements_par_livraison
    .groupBy("nb_evenements")
    .count()
    .withColumnRenamed("count", "nb_livraisons")
    .orderBy("nb_evenements")
    .show()
)

nb_livraisons_plusieurs_clients = (
    evenements_par_livraison
    .filter(F.col("nb_clients") >= 2)
    .count()
)

print(
    "Livraisons concernant au moins deux clients différents : "
    f"{nb_livraisons_plusieurs_clients}"
)

+-------------+-------------+
|nb_evenements|nb_livraisons|
+-------------+-------------+
|            1|       217538|
|            2|        24873|
|            3|         3053|
|            4|          592|
|            5|          155|
|            6|           29|
|            7|            8|
|            8|            5|
|            9|            4|
|           10|            1|
+-------------+-------------+



Livraisons concernant au moins deux clients différents : 12536


5.8) Ajoutez une colonne `duree_ev_min` à chaque événement : le temps écoulé depuis l'événement précédent de la même livraison, ou depuis le début de la livraison pour le premier. Quelle est la durée moyenne d'une remise signée ? D'un échec ?

In [54]:
fenetre_evenements = (
    Window
    .partitionBy("livreur_id", "jour", "numero_livraison")
    .orderBy("signature")
)

livraisons_ev = livraisons_ev.withColumn(
    "signature_precedente",
    F.lag("signature", 1).over(fenetre_evenements)
)

livraisons_ev = (
    livraisons_ev
    .withColumn(
        "duree_ev_min",
        (
            F.col("signature").cast("long")
            - F.coalesce(
                F.col("signature_precedente"),
                F.col("debut")
            ).cast("long")
        ) / 60
    )
    .drop("signature_precedente")
)

(
    livraisons_ev
    .filter(F.col("evenement").isin("livree", "echec"))
    .groupBy("evenement")
    .agg(
        F.round(F.avg("duree_ev_min"), 2).alias("duree_moyenne_min")
    )
    .orderBy("evenement")
    .show()
)

+---------+-----------------+
|evenement|duree_moyenne_min|
+---------+-----------------+
|    echec|             2.02|
|   livree|             4.24|
+---------+-----------------+



5.9) Pour chaque livreur : nombre de livraisons signées, nombre d'échecs, nombre de jours travaillés, livraisons signées par jour. Affichez les 10 livreurs qui signent le plus de livraisons par jour.

In [ ]:
bilan_evenements = (
    commandes
    .groupBy("livreur_id")
    .agg(
        F.count(
            F.when(F.col("evenement") == "livree", 1)
        ).alias("nb_signatures"),
        F.count(
            F.when(F.col("evenement") == "echec", 1)
        ).alias("nb_echecs")
    )
)

jours_travailles = (
    gps
    .filter(F.col("livreur_id").isNotNull())
    .groupBy("livreur_id")
    .agg(
        F.countDistinct("jour").alias("nb_jours_travailles")
    )
)

bilan_livreurs = (
    jours_travailles
    .join(bilan_evenements, on="livreur_id", how="left")
    .fillna(0, subset=["nb_signatures", "nb_echecs"])
    .withColumn(
        "signatures_par_jour",
        F.col("nb_signatures") / F.col("nb_jours_travailles")
    )
)

(
    bilan_livreurs
    .orderBy(
        F.col("signatures_par_jour").desc(),
        F.col("livreur_id")
    )
    .select(
        "livreur_id",
        "nb_signatures",
        "nb_echecs",
        "nb_jours_travailles",
        F.round("signatures_par_jour", 2).alias("signatures_par_jour")
    )
    .show(10, truncate=False)
)

## 6) Les distances

6.1) Ajoutez à `gps` une colonne `distance_km` : la distance depuis le point précédent du même livreur, le même jour (`haversine`). Le premier point de la journée vaut 0.

*Un point par minute coupe les virages : la distance calculée sous-estime d'environ 12 % la distance réellement parcourue.*

6.2) Kilomètres parcourus sur l'année par chaque véhicule : les 10 premiers, et le total de la flotte.

6.3) Kilomètres parcourus sur l'année par chaque livreur : les 10 premiers.

6.4) Kilomètres parcourus par jour : moyenne par véhicule-jour, pour chaque entrepôt.

6.5) Distance entre deux livraisons : la somme de `distance_km` de chaque (`livreur_id`, `jour`, `numero_livraison`), c'est-à-dire le trajet qui suit la livraison jusqu'à la suivante. Moyenne et médiane, pour les livraisons de type `livraison`.

## 7) Les ventes

7.1) Chiffre d'affaires par mois (mois de la commande).

7.2) Panier moyen : montant moyen d'une commande.

7.3) Panier médian.

7.4) Top 10 des produits par chiffre d'affaires.

7.5) Top 10 des produits par quantité vendue.

7.6) Top 10 des clients par dépense sur l'année.

7.7) Top 10 des clients par nombre de commandes.

## 8) La qualité de service

8.1) Délai entre la commande et sa livraison, en jours calendaires (date de la signature − date de la commande). Combien de commandes livrées pour chaque délai ?

8.2) Part des commandes livrées le lendemain, pour chaque mois de commande.

8.3) Construisez la table `suivi` : une ligne par commande, avec `nb_tentatives` (événements `livree` + `echec`), `nb_echecs` et `issue` (`livree`, `retour` ou `en cours`). Combien de commandes pour chaque nombre de tentatives ?

8.4) Motifs d'échec : nombre d'échecs par motif.

8.5) Top 10 des clients au plus fort taux d'échec (échecs / tentatives), parmi les clients qui ont au moins 10 tentatives.

## 9) Les clients

On construit la table `clients` colonne par colonne : une ligne par client qui a passé au moins une commande.

9.1) Construisez la table `clients` : `client_id`, `nb_commandes`, `ca` (somme des montants), `panier_moyen`.

9.2) Ajoutez `premiere_commande` et `derniere_commande` (dates).

9.3) Ajoutez `lon` et `lat` : la position moyenne des livraisons où le client a signé. Combien de clients n'ont aucune position ?

9.4) Ajoutez `commune` et `code_postal` : la commune dont le centre (`communes.csv`) est le plus proche de la position du client.

9.5) Les 10 communes qui passent le plus de commandes, avec leur chiffre d'affaires.

9.6) Ajoutez `tentatives`, `echecs` et `taux_echec` (à partir de `suivi`, 8.3).

9.7) Ajoutez `categorie_dominante` (la catégorie où le client dépense le plus) et `part_dominante` (part de sa dépense dans cette catégorie).

9.8) Ajoutez `unites_par_ligne` : la quantité moyenne par ligne de commande.

9.9) Segments de dépense annuelle (`ca`) : moins de 1 000 €, 1 000 à 5 000 €, 5 000 à 20 000 €, 20 000 à 50 000 €, 50 000 € et plus. Ajoutez la colonne `segment`, puis pour chaque segment : nombre de clients, chiffre d'affaires, chiffre d'affaires moyen par client, part du chiffre d'affaires total.

9.10) Nouveaux clients par mois (mois de la première commande), en tableau et en graphique.

9.11) Pour chaque catégorie dominante : nombre de clients et chiffre d'affaires.

9.12) Clients atypiques : les 10 clients dont `unites_par_ligne` est la plus élevée, avec le rapport à la moyenne de tous les clients, leur `ca` et leur `categorie_dominante`.

## 10) Data visualisation

Échelles de couleur : blanc ou clair pour 0, foncé pour les valeurs élevées.

10.1) Heatmap jour de la semaine × heure du nombre de commandes passées.

10.2) Carte du nombre de livraisons signées par commune ou arrondissement (`communes.geojson`, carte choroplèthe).

10.3) Heatmap heure × mois de la vitesse moyenne quand le livreur est à bord et que le véhicule roule.

10.4) Treemap du chiffre d'affaires : catégorie → sous-catégorie.

## 11) Sauvegarde et modélisation

11.2) **Modélisation.** Les analyses précédentes recalculent les mêmes jointures et agrégations. Proposez des collections MongoDB qui rendent ces questions immédiates : pour chaque collection, ce que représente un document, ses champs, ses tableaux imbriqués, et les questions des parties 5 à 9 qu'elle simplifie. Écrivez votre proposition dans la cellule ci-dessous.

*Votre proposition :*